# Lesson 3 — Counts to probabilities (MLE)

Turning counts into probabilities is a single division: P(w | context) = count(context, w) / count(context).
That's the maximum-likelihood estimate. There's no training loop: counting *is* the learning. At the end I have a
`suggest_next()` autocomplete.

## Same tools as Lesson 2

The same tokenizer and counter, but now I load the corpus from the `corpus.txt` I saved last lesson.

In [1]:
import re, math, random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """context (n-1 words) -> Counter of the words that follow it."""
    foll = defaultdict(Counter)
    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]
        for i in range(len(toks) - n + 1):
            foll[tuple(toks[i:i + n - 1])][toks[i + n - 1]] += 1
    return foll

In [4]:
sents = to_sentences(open("corpus.txt").read())
foll = build(sents, n=2)

## Probability = pair count / context count

`prob` divides how often `word` followed `context_word` by how often `context_word` appeared as a context at all
(returning 0 if it never did).

In [5]:
def prob(context_word, word):
    c = foll[(context_word,)]; total = sum(c.values())
    return c[word] / total if total else 0.0

`suggest_next` does the same division for the `k` most common followers, giving a ranked suggestion bar.

In [6]:
def suggest_next(context_word, k=3):
    c = foll[(context_word,)]; total = sum(c.values())
    return [(w, round(n / total, 3)) for w, n in c.most_common(k)]

In [7]:
print("P(coffee | morning) =", round(prob("morning", "coffee"), 3))
print("P(train  | morning) =", round(prob("morning", "train"), 3))
print()
for w in ["i", "the", "morning", "good", "drink"]:
    print(f"after {w!r:9s}->", suggest_next(w))

P(coffee | morning) = 0.125
P(train  | morning) = 0.125

after 'i'      -> [('drink', 0.125), ('read', 0.125), ('walk', 0.125)]
after 'the'    -> [('morning', 0.103), ('weekend', 0.088), ('park', 0.074)]
after 'morning'-> [('</s>', 0.5), ('train', 0.125), ('coffee', 0.125)]
after 'good'   -> [('book', 0.6), ('coffee', 0.4)]
after 'drink'  -> [('coffee', 0.5), ('tea', 0.5)]


"morning" appears 16 times as a context, and "coffee" follows it twice: 2/16 = 0.125. The suggester works,
but it only ever sees one word back, so its guesses are shallow. Next lesson lets it write.

## Test yourself

**1. How is the MLE probability P(w | context) computed?**
count(context, w) divided by count(context). It's how often w followed that context, out of every time the context
appeared. Nothing is trained by gradient descent.

**2. Why is P(coffee | morning) = 0.125?**
"morning" appears 16 times as a context (it ends 8 sentences and is followed by something else 8 times), and
"coffee" follows it twice: 2/16 = 0.125.

**3. What is the weakness of a bigram suggester?**
It only knows the last word, so "i drink" and "they drink" get the same suggestions. It also gives exactly zero
to any pair it never saw.

## My notes

The key idea: "learning" a bigram model is just counting and dividing. What surprised me is that `</s>` comes out
as the top "suggestion" after "morning". A real keyboard would have to filter it out.